# Pipeline SOME/IP: PCAP → Features → t-SNE

Reprodução de **Kim et al. (2026)** — Seções 5.1, 5.2, 5.3 e 6.1.

| Etapa | Script | Saída |
|-------|--------|-------|
| 1 — Parse PCAPs | `01_parse_pcap.py` | `parsed_packets.csv` |
| 2 — Extrai features | `02_extract_features.py` | `train_features.csv`, `test_features.csv` |
| 3 — t-SNE | este notebook | Figura 9 (proposed vs header-based) |

> **Pré-requisito:** arquivos `.pcap` do Figshare na pasta configurada em `PCAP_DIR`.

## 1. Configuração

In [1]:
import sys, time
import numpy as np
import pandas as pd
from pathlib import Path

# ── Ajuste estes caminhos ──────────────────────────────────────────────────
PCAP_DIR    = Path('data/pcap')       # pasta com os .pcap do Figshare
DATA_DIR    = Path('data')            # pasta de saída dos CSVs gerados
SCRIPTS_DIR = Path('../files')        # pasta com 01_parse_pcap.py etc.
# ──────────────────────────────────────────────────────────────────────────

PARSED_CSV = DATA_DIR / 'parsed_packets.csv'
TRAIN_CSV  = DATA_DIR / 'train_features.csv'
TEST_CSV   = DATA_DIR / 'test_features.csv'

DATA_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SCRIPTS_DIR.resolve()))

print('Caminhos:')
print(f'  PCAPs  : {PCAP_DIR.resolve()}')
print(f'  Dados  : {DATA_DIR.resolve()}')
print(f'  Scripts: {SCRIPTS_DIR.resolve()}')
print()
pcaps = sorted(PCAP_DIR.glob('*.pcap')) if PCAP_DIR.exists() else []
print(f'PCAPs encontrados: {len(pcaps)}')
for p in pcaps:
    print(f'  {p.name}  ({p.stat().st_size/1e6:.0f} MB)')
if not pcaps:
    print('  [AVISO] Nenhum .pcap encontrado — verifique PCAP_DIR')


Caminhos:
  PCAPs  : C:\Mestrado\SDV_Research\experiments\notebooks\data\pcap
  Dados  : C:\Mestrado\SDV_Research\experiments\notebooks\data
  Scripts: C:\Mestrado\SDV_Research\experiments\files

PCAPs encontrados: 0
  [AVISO] Nenhum .pcap encontrado — verifique PCAP_DIR


## 2. Etapa 1 — Parsing dos PCAPs

Extrai campos IP / TCP / UDP / SOME/IP de cada pacote e salva como CSV.  
**Pula automaticamente** se `parsed_packets.csv` já existir.

In [ ]:
if PARSED_CSV.exists():
    print(f'[OK] {PARSED_CSV.name} ja existe ({PARSED_CSV.stat().st_size/1e6:.0f} MB) — pulando Etapa 1.')
else:
    import importlib.util, types
    spec = importlib.util.spec_from_file_location('parse_pcap', SCRIPTS_DIR / '01_parse_pcap.py')
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    print('Iniciando parsing... (10-30 min)')
    t0 = time.time()
    mod.process_all_pcaps(str(PCAP_DIR), str(PARSED_CSV))
    print(f'Concluido em {(time.time()-t0)/60:.1f} min')


### Inspeciona CSV parseado

In [ ]:
label_counts = pd.read_csv(PARSED_CSV, usecols=['label'])['label'].value_counts()
total = label_counts.sum()
print(f'Total de registros SOME/IP: {total:,}')
print()
for lbl, cnt in label_counts.items():
    print(f'  {lbl:<35} {cnt:>10,}  ({100*cnt/total:.1f}%)')

print()
df_head = pd.read_csv(PARSED_CSV, nrows=3)
print(f'Colunas: {list(df_head.columns)}')


## 3. Etapa 2 — Extração das 9 Features Comportamentais

Calcula as features da Tabela 1 por fluxo (five-tuple).  
**Pula automaticamente** se `train_features.csv` já existir.

In [ ]:
if TRAIN_CSV.exists() and TEST_CSV.exists():
    print(f'[OK] train/test_features.csv ja existem — pulando Etapa 2.')
else:
    import importlib.util
    spec = importlib.util.spec_from_file_location('extract_features', SCRIPTS_DIR / '02_extract_features.py')
    mod2  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod2)
    print('Extraindo features... (5-20 min)')
    t0 = time.time()
    mod2.run_feature_extraction(str(PARSED_CSV), str(DATA_DIR))
    print(f'Concluido em {(time.time()-t0)/60:.1f} min')


### Carrega features e verifica distribuição

In [ ]:
FEATURE_COLS = [
    'f01_ip_time_interval', 'f02_someip_likelihood', 'f03_tcpudp_likelihood',
    'f04_someip_entropy',   'f05_tcpudp_entropy',
    'f06_someip_payload_changes', 'f07_tcpudp_payload_changes',
    'f08_ip_length_changes', 'f09_tcpudp_length_changes',
]

df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f'Treino : {len(df_train):>8,}  | Normal: {(df_train.label==0).sum():,}  Ataque: {(df_train.label==1).sum():,}')
print(f'Teste  : {len(df_test):>8,}  | Normal: {(df_test.label==0).sum():,}  Ataque: {(df_test.label==1).sum():,}')
print()
display(df_test[FEATURE_COLS].describe().round(4))


## 4. Análise t-SNE — Seção 6.1

Compara dois conjuntos de features:
- **(a) Proposed**: 9 features comportamentais (f01–f09)
- **(b) Header-based + Proposed**: campos numéricos IP/TCP/UDP/SOME/IP + f01–f09

Amostra: **5.000 normais + 5.000 ataques** do conjunto de teste (igual ao artigo).

### 4.1 Amostragem balanceada

In [ ]:
rng = np.random.default_rng(42)

y_all = df_test['label'].values
idx_n = np.where(y_all == 0)[0]
idx_a = np.where(y_all == 1)[0]
n_samp = min(5000, len(idx_n), len(idx_a))

sel_n = rng.choice(idx_n, size=n_samp, replace=False)
sel_a = rng.choice(idx_a, size=n_samp, replace=False)
sel   = np.concatenate([sel_n, sel_a])
rng.shuffle(sel)

df_s  = df_test.iloc[sel].reset_index(drop=True)
y_s   = df_s['label'].values
X_prop = df_s[FEATURE_COLS].fillna(0).values

print(f'Amostra: {(y_s==0).sum()} normais + {(y_s==1).sum()} ataques = {len(y_s):,} total')


### 4.2 Features de cabeçalho (header-based)

In [ ]:
# Campos numéricos do cabeçalho — gerados pelo 01_parse_pcap.py
# e propagados pelo 02_extract_features.py
HEADER_CANDIDATE = [
    'ip_ttl', 'ip_len', 'ip_id', 'ip_flags',
    'src_port', 'dst_port', 'transport_len',
    'tcp_seq', 'tcp_ack', 'tcp_flags',
    'service_id', 'method_id', 'someip_len',
    'client_id', 'session_id', 'proto_ver', 'iface_ver',
    'msg_type', 'return_code', 'someip_payload_len',
]

HDR_COLS = [c for c in HEADER_CANDIDATE if c in df_test.columns]
print(f'Colunas de cabecalho disponiveis: {len(HDR_COLS)}')
print(f'  {HDR_COLS}')

if HDR_COLS:
    X_hdr  = df_s[HDR_COLS].fillna(0).values
    X_comb = np.hstack([X_hdr, X_prop])   # header + proposed
    print(f'Combined shape: {X_comb.shape}')
else:
    X_comb = None
    print('[AVISO] Sem colunas de cabecalho no CSV — so proposed sera plotado.')


### 4.3 Normalização e t-SNE

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.manifold import TSNE

X_prop_sc = MinMaxScaler().fit_transform(X_prop)

print('t-SNE: proposed features...')
t0 = time.time()
X2d_prop = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42).fit_transform(X_prop_sc)
print(f'  {time.time()-t0:.0f}s')

X2d_comb = None
if X_comb is not None:
    X_comb_sc = MinMaxScaler().fit_transform(X_comb)
    print('t-SNE: header-based + proposed...')
    t0 = time.time()
    X2d_comb = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42).fit_transform(X_comb_sc)
    print(f'  {time.time()-t0:.0f}s')


### 4.4 Métricas de clustering

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

def fisher_ratio(X2d, y):
    mu0  = X2d[y == 0].mean(axis=0)
    mu1  = X2d[y == 1].mean(axis=0)
    var0 = X2d[y == 0].var(axis=0).sum()
    var1 = X2d[y == 1].var(axis=0).sum()
    return float(np.sum((mu1 - mu0) ** 2) / (var0 + var1 + 1e-12))

def show_metrics(X2d, y, name):
    sil = silhouette_score(X2d, y)
    ch  = calinski_harabasz_score(X2d, y)
    db  = davies_bouldin_score(X2d, y)
    fr  = fisher_ratio(X2d, y)
    print(f'{name}')
    print(f'  Silhouette Score  : {sil:.4f}  (higher=better)')
    print(f'  Calinski-Harabasz : {ch:.1f}  (higher=better)')
    print(f'  Davies-Bouldin    : {db:.4f}  (lower=better)')
    print(f'  Fisher Ratio      : {fr:.4f}  (higher=better)')
    return dict(silhouette=sil, ch=ch, db=db, fisher=fr)

m_prop = show_metrics(X2d_prop, y_s, 'Proposed (behavioral features)')
print()
m_comb = None
if X2d_comb is not None:
    m_comb = show_metrics(X2d_comb, y_s, 'Header-based + Proposed')
    print()
    print('Artigo (Sec. 6.1): header-based silhouette = 0.0721')


### 4.5 Figura 9 — Plot

In [ ]:
import matplotlib.pyplot as plt

def tsne_ax(ax, X2d, y, title, metrics):
    for cls, color, lbl in [(0, '#2196F3', 'Normal'), (1, '#F44336', 'Attack')]:
        m = y == cls
        ax.scatter(X2d[m,0], X2d[m,1], c=color, label=lbl,
                   alpha=0.25, s=4, rasterized=True)
    ax.set_title(title, fontsize=11)
    info = (f'Sil={metrics["silhouette"]:.4f}  '
            f'CH={metrics["ch"]:.0f}  '
            f'DB={metrics["db"]:.4f}  '
            f'Fisher={metrics["fisher"]:.4f}')
    ax.set_xlabel(info, fontsize=8)
    ax.set_ylabel('t-SNE dim 2')
    ax.legend(markerscale=3, fontsize=8, loc='best')

n_plots = 2 if X2d_comb is not None else 1
fig, axes = plt.subplots(1, n_plots, figsize=(7*n_plots, 6))
if n_plots == 1:
    axes = [axes]

tsne_ax(axes[0], X2d_prop, y_s,
        '(a) Proposed Feature Set\n(9 behavioral)', m_prop)

if X2d_comb is not None:
    tsne_ax(axes[1], X2d_comb, y_s,
            '(b) Header-based + Proposed', m_comb)

plt.suptitle('Figure 9 — t-SNE (Kim et al. 2026, Sec. 6.1)', fontsize=12)
plt.tight_layout()
plt.savefig('figure9_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: figure9_tsne.png')
